# Load and initialise

In [ ]:
import sys
sys.path.append('..')

import torch
from tqdm import tqdm
from models import GenTeacher, GenStudent, GenClassifier, vit_parameters, classifier_layels_list
from models.model import AttentionAutoEncoder
from utils import save_model
from utils.dataset import get_datasets, image_transform
from utils.channels import Channels

device = 'cuda:0'
student = GenStudent(vit_parameters, device)
teacher = GenTeacher('ViT-B/32', device)
teacher.load_state_dict(torch.load('../results/weights/retrain_teacher/cifar100_teacher_model_sota.pt'))
classifier = GenClassifier(512, 100, classifier_layels_list, device)
atten_ae = AttentionAutoEncoder(512, 2).to(device)
train_loader, test_loader = get_datasets(image_transform, 500, 'cifar100')

# Train

In [ ]:
student.load_state_dict(torch.load('../results/weights/distilltion/kd_student_model_sota.pt'))
classifier.load_state_dict(torch.load('../results/weights/distilltion/kd_classifier_student_model_sota.pt'))

channel = Channels(device)
optimizer_cla = torch.optim.Adam(classifier.parameters(), lr=1e-3)
optimizer_ae = torch.optim.Adam(atten_ae.parameters(), lr=1e-3)

criterion_AE = torch.nn.MSELoss()
criterion_CE = torch.nn.CrossEntropyLoss()

student.eval()
teacher.eval()
classifier.train()
atten_ae.train()

ACC_AE = []
LOSS_AE = []
TRAIN_SNR_LIST = [-5, -2, 0, 5, 10, 15, 20]

for epoch, snr in enumerate(TRAIN_SNR_LIST):
    train_loader = tqdm(train_loader,leave=False, ncols=120)
    # 均匀采样snr
    # snr = torch.randint(-5, 20, (1,)).item()
    # snr = random.choice(SNR)
    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)

        optimizer_cla.zero_grad()
        optimizer_ae.zero_grad()

        features_stu = student(images)
        feather_tea = teacher(images)

        latten_ae = atten_ae.encoder(features_stu)
        latten_ae = channel.AWGN(latten_ae, snr)
        features_hat = atten_ae.decoder(latten_ae)

        pre = classifier(features_hat)

        loss_AE = criterion_AE(features_hat, feather_tea)
        loss_CE = criterion_CE(pre, labels)

        loss = 0.5 * loss_AE + 0.5 * loss_CE

        acc = (pre.argmax(dim=1) == labels).float().mean()
        loss.backward()

        optimizer_cla.step()
        optimizer_ae.step()
        
        ACC_AE.append(acc.item())
        LOSS_AE.append(loss.item())

        train_loader.set_description(f'Epoch:{epoch+1} Training on SNR={snr}')
        train_loader.set_postfix(loss=loss_AE.item(),acc = acc.item())

# Test at different SNRs

In [ ]:
student.eval()
classifier.eval()
atten_ae.eval()
channels = Channels(device)

ACC_with_SNR = []
for snr in range(-10, 25, 2):
    test_loader = tqdm(test_loader, desc='Testing on SNR:'+str(snr), leave = False, ncols=120)
    Length = 0
    acc = 0
    for i, (images, labels) in enumerate(test_loader):
        images, labels = images.to(device), labels.to(device)

        with torch.no_grad():
            features = student(images)
            features = atten_ae.encoder(features)
            features_awgn = channels.AWGN(features, snr)
            features_hat = atten_ae.decoder(features_awgn)
            logits = classifier(features_hat)

            preds = logits.argmax(dim=1)
            acc += (preds == labels).float().sum()

        Length = (i+1)*len(labels)

        test_loader.set_postfix(Acc = acc.item() / Length)
        test_loader.update()
    ACC_with_SNR.append(acc.item() / Length)

print(ACC_with_SNR)

# Save model weights

In [ ]:
save_model(classifier, '../results/weights/joint_training_with_ae/classifier_ae.pt')
save_model(atten_ae, '../results/weights/joint_training_with_ae/atten_ae.pt')